# 📄 Simple RAG Lab (Free, No API Key)
Upload a PDF, build a RAG index, and ask questions. Uses open-source models running locally, so there are **no token limits and no cost**.

**Pipeline:** PDF → text → chunks → embeddings → retrieve → LLM answer

> Tip: *Runtime → Change runtime type → T4 GPU* makes it faster (optional).

## 1. Install dependencies

In [ ]:
!pip install -q pypdf sentence-transformers transformers torch numpy

## 2. Load models

In [ ]:
import numpy as np
import torch
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
LLM_MODEL = "google/flan-t5-base"   # try "google/flan-t5-large" on a GPU for better answers

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

embedder = SentenceTransformer(EMBED_MODEL, device=device)
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
llm = AutoModelForSeq2SeqLM.from_pretrained(LLM_MODEL).to(device)

## 3. RAG functions

In [ ]:
def read_pdf(path):
    """Extract all text from a PDF."""
    reader = PdfReader(path)
    return "\n".join((p.extract_text() or "") for p in reader.pages)


def chunk_text(text, chunk_words=200, overlap=40):
    """Split text into overlapping word-based chunks."""
    words = text.split()
    step = chunk_words - overlap
    chunks = [" ".join(words[i:i + chunk_words]) for i in range(0, len(words), step)]
    return [c for c in chunks if c]


def build_index(chunks):
    """Embed every chunk (the 'training'/indexing step of RAG)."""
    return embedder.encode(chunks, normalize_embeddings=True, show_progress_bar=True)


def retrieve(question, chunks, embeddings, k=3):
    """Return the top-k chunks most similar to the question."""
    q = embedder.encode([question], normalize_embeddings=True)[0]
    top = np.argsort(embeddings @ q)[::-1][:k]
    return [chunks[i] for i in top]


def answer(question, chunks, embeddings, k=3, show_context=False):
    """Retrieve context, then let the LLM answer using only that context."""
    context = "\n\n".join(retrieve(question, chunks, embeddings, k))
    prompt = (
        "Answer the question using only the context below. "
        "If the answer is not in the context, say \"I don't know\".\n\n"
        f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(device)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=256, num_beams=4)
    reply = tokenizer.decode(out[0], skip_special_tokens=True)
    if show_context:
        print("--- Retrieved context ---\n", context, "\n-------------------------")
    return reply

## 4. Upload your PDF and build the RAG

In [ ]:
from google.colab import files

print("Please upload a PDF file...")
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]

text = read_pdf(pdf_path)
if not text.strip():
    raise SystemExit("No text found. The PDF may be a scanned image (needs OCR).")

chunks = chunk_text(text)
print(f"Created {len(chunks)} chunks. Building index...")
embeddings = build_index(chunks)
print("✅ RAG is ready!")

## 5. Ask questions
Run this cell as many times as you like. Type `quit` to stop.

In [ ]:
while True:
    q = input("Your question (or 'quit'): ").strip()
    if q.lower() in {"quit", "exit", ""}:
        break
    print("Answer:", answer(q, chunks, embeddings), "\n")

## 6. (Optional) One-off question with retrieved context shown

In [ ]:
print(answer("What is this document about?", chunks, embeddings, show_context=True))